In [1]:
import pandas as pd
from datetime import datetime
import os

In [2]:
# Load the CSV file into a pandas DataFrame
cPath = os.path.dirname(os.path.realpath('.'))
file_path = cPath + '/KVA_based/Data/Combined_Terminal.csv'
print(file_path)
#allData= pd.read_csv(file_path)
allData= pd.read_csv(file_path, parse_dates=['DateTime'], date_format='%Y-%m-%d %H:%M:%S')
allData.set_index('DateTime', drop = False, inplace=True)
print(allData.head)
print(allData.index.name)
allData['Delta'] = 0
# Adding a 'label' column to differentiate between OffPeak, OnPeak, and OnPeakHigh 
allData['label'] = 'OffPeak'
allData.head
print(allData['DateTime'][0], allData['DateTime'][1])
print(allData['KWH11'][0], allData['KWH11'][1])

/home/homayoun-valafar/Dropbox/Projects/VisualCode/Python/CAE/New/KVA_based/Data/Combined_Terminal.csv
<bound method NDFrame.head of                                DateTime  KWH11  KVAH71    KW  Year
DateTime                                                          
2014-07-01 00:15:00 2014-07-01 00:15:00  357.5  401.84  1430  2014
2014-07-01 00:30:00 2014-07-01 00:30:00  362.0  407.68  1448  2014
2014-07-01 00:45:00 2014-07-01 00:45:00  350.5  394.71  1402  2014
2014-07-01 01:00:00 2014-07-01 01:00:00  344.5  388.46  1378  2014
2014-07-01 01:15:00 2014-07-01 01:15:00  324.5  364.04  1298  2014
...                                 ...    ...     ...   ...   ...
2024-12-31 23:00:00 2024-12-31 23:00:00  223.5  250.89   894  2024
2024-12-31 23:15:00 2024-12-31 23:15:00  198.5  219.21   794  2024
2024-12-31 23:30:00 2024-12-31 23:30:00  211.0  235.02   844  2024
2024-12-31 23:45:00 2024-12-31 23:45:00  219.0  245.07   876  2024
2025-01-01 00:00:00 2025-01-01 00:00:00  194.0  212.82   776  2

/tmp/ipykernel_14982/3151361149.py:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(allData['DateTime'][0], allData['DateTime'][1])
/tmp/ipykernel_14982/3151361149.py:15: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print(allData['KWH11'][0], allData['KWH11'][1])


In [3]:
# Find irregular data
def missing_data(data):
    e_td = pd.to_timedelta('15 minutes')
    for i in range(0, len(data)-1, 1):
        c_td = data['DateTime'][i+1] - data['DateTime'][i]
        if(e_td != c_td): 
           print (data['DateTime'][i], data['DateTime'][i+1], c_td)

missing_data(allData)

/tmp/ipykernel_14982/3611234131.py:5: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  c_td = data['DateTime'][i+1] - data['DateTime'][i]
/tmp/ipykernel_14982/3611234131.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  print (data['DateTime'][i], data['DateTime'][i+1], c_td)


2014-11-02 01:45:00 2014-11-02 01:00:00 -1 days +23:15:00
2015-03-08 01:45:00 2015-03-08 03:00:00 0 days 01:15:00
2015-11-01 01:45:00 2015-11-01 01:00:00 -1 days +23:15:00
2016-03-13 01:45:00 2016-03-13 03:00:00 0 days 01:15:00
2016-11-06 01:45:00 2016-11-06 01:00:00 -1 days +23:15:00
2017-03-12 01:45:00 2017-03-12 03:00:00 0 days 01:15:00
2017-11-05 01:45:00 2017-11-05 01:00:00 -1 days +23:15:00
2018-03-11 01:45:00 2018-03-11 03:00:00 0 days 01:15:00
2018-11-04 01:45:00 2018-11-04 01:00:00 -1 days +23:15:00
2019-03-10 01:45:00 2019-03-10 03:00:00 0 days 01:15:00
2019-11-03 01:45:00 2019-11-03 01:00:00 -1 days +23:15:00
2020-03-08 01:45:00 2020-03-08 03:00:00 0 days 01:15:00
2020-11-01 01:45:00 2020-11-01 01:00:00 -1 days +23:15:00
2021-03-14 01:45:00 2021-03-14 03:00:00 0 days 01:15:00
2021-11-07 01:45:00 2021-11-07 01:00:00 -1 days +23:15:00
2022-03-13 01:45:00 2022-03-13 03:00:00 0 days 01:15:00
2022-11-06 01:45:00 2022-11-06 01:00:00 -1 days +23:15:00
2023-03-12 01:45:00 2023-03-12

In [4]:
# Calculate the derivative of the usage
def calc_delta(data):
    e_td = pd.to_timedelta('15 minutes')
    delta = []
    for i in range(0, len(data)-1, 1):
        dUsage = data['KVAH71'][i+1] - data['KVAH71'][i]
        dt = data['DateTime'][i+1] - data['DateTime'][i]
        delta.append(dUsage / dt.total_seconds())
    delta.append(0)
    return delta 

allData['Delta'] = calc_delta(allData)

/tmp/ipykernel_14982/2868394588.py:6: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dUsage = data['KVAH71'][i+1] - data['KVAH71'][i]
/tmp/ipykernel_14982/2868394588.py:7: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dt = data['DateTime'][i+1] - data['DateTime'][i]


In [5]:
# allData.describe()
allData['KVAH71'].describe()

count    368264.000000
mean        239.180075
std          64.936135
min           0.000000
25%         190.470000
50%         230.240000
75%         280.690000
max         504.080000
Name: KVAH71, dtype: float64

In [6]:
# This needs closer inspection, but it seems to be a way to set the labels based on the month and hour

def set_labels(data):
    data['label'] = 'w'  # Default label as winter
    month_mask = data.index.month.isin([6, 7, 8, 9])  # June to September
    data.loc[month_mask, 'label'] = 's' # Summer months

    # Identify OnPeak hours in summer
    # Modifying it to exclude 9PM
    time_mask = data.index.hour.isin([13, 14, 15, 16, 17, 18, 19, 20])  # Summer OnPeak hours. Does not include 22
    label_mask = data['label'] == 's'
    data.loc[time_mask & label_mask, 'label'] = 'sOnPeak'  # Summer OnPeak

    # Identify OnPeak hours in summer transition months (May and October) 
    time_mask = data.index.hour.isin([13, 14, 15, 16, 17, 18, 19, 20, 21])  # OnPeak hours. Does not include 22
    month_mask = data.index.month.isin([5, 10])  # May or October
    label_mask = data['label'] == 'w'
    data.loc[time_mask & label_mask & month_mask, 'label'] = 'wOnPeak'  # Winter OnPeak

    # Identify OnPeak hours on deep winter months (November through April) 
    # Modifying it to exclude 9PM
    time_mask = data.index.hour.isin([6, 7, 8, 9, 10, 11, 17, 18, 19, 20])  # OnPeak hours. Does not include 22
    # month_mask = data.index.month.isin([5, 10])  # Not needed for the last modification. All remaining are w 
    label_mask = data['label'] == 'w'
    data.loc[time_mask & label_mask, 'label'] = 'wOnPeak'  # Winter OnPeak

    # Identify the OffPeak times
    label_mask = data['label'] == 's'
    data.loc[label_mask, 'label'] = 'sOffPeak'  # Summer OffPeak 
    label_mask = data['label'] == 'w'
    data.loc[label_mask, 'label'] = 'wOffPeak'  # Winter OffPeak
    label_mask = data.index.day_of_week.isin([5, 6])  # Saturday and Sunday
    data.loc[label_mask, 'label'] = 'wkndOffPeak'  # Weekend OffPeak

    #Identify the high usage periods during OnPeak times
    data.loc[(data['label'] == 'wOnPeak') & (data['KWH11'] > 200), 'label'] = 'wOnPeakHigh'
    data.loc[(data['label'] == 'sOnPeak') & (data['KWH11'] > 200), 'label'] = 'sOnPeakHigh'
    return data

allData = set_labels(allData)
print('allData: ', allData.value_counts('label'), '\n')

allData:  label
wkndOffPeak    105219
wOffPeak        91638
wOnPeak         62899
sOffPeak        60031
sOnPeak         19164
wOnPeakHigh     18461
sOnPeakHigh     10852
Name: count, dtype: int64 



In [7]:
allData.to_csv('Combined_Terminal_Label.csv', index=False)